In [2]:
import scanpy as sc
import os

# ==============================================================================
# 1. Configuration & File Paths
# ==============================================================================
input_file = "../data/processed/pca_computed.h5ad"
output_file = "../data/processed/clustered.h5ad"

# Directory to save the final visualization images
figure_dir = "../results/figures/"
os.makedirs(figure_dir, exist_ok=True)

# Scanpy settings for high-quality figure output
sc.settings.set_figure_params(dpi=300, facecolor='white', format='png')
sc.settings.figdir = figure_dir

In [ ]:
# These hyperparameters control the clustering and dimensionality reduction steps.
# You can experiment with these values to see how they affect the results. The chosen values 
# are commonly used defaults for PBMC datasets.

# - `n_neighbors`: Determines how many nearest neighbors are considered when constructing the 
# neighborhood graph.

# - `n_pcs`: The number of principal components to use from the PCA step. This can affect the
# quality of the clustering and visualization.

# - `resolution`: Controls the granularity of the clustering. Higher values will result in more
# clusters, while lower values will yield fewer clusters. The optimal value can depend on the   
# dataset and the biological question being asked.

# Hyperparameters
n_neighbors = 10  # Number of local neighbors to consider for the graph
n_pcs = 40        # Number of Principal Components to use from the previous step
resolution = 0.5  # Controls how granular the clustering is (higher = more clusters)

In [4]:
# ==============================================================================
# 2. Execution
# ==============================================================================
print(f"Loading PCA data from {input_file}...")
adata = sc.read_h5ad(input_file)

Loading PCA data from ../data/processed/pca_computed.h5ad...


In [ ]:
# ==============================================================================
# 3. Compute Neighborhood Graph
# ==============================================================================
# Before clustering, we build a mathematical graph showing which cells are 
# closest to each other in the high-dimensional PCA space.

# This step is crucial because it forms the basis for the clustering and UMAP visualization.
# The neighborhood graph captures the local structure of the data, allowing us to identify 
# groups of similar cells.
# The `n_neighbors` parameter determines how many nearest neighbors are considered when constructing
# the graph, while `n_pcs` specifies how many principal components to use from the PCA step.

print(f"Computing neighborhood graph using {n_pcs} PCs and {n_neighbors} neighbors...")
sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)

In [ ]:
# ==============================================================================
# 4. Clustering (Leiden Algorithm)
# ==============================================================================
# Leiden is the modern industry standard for single-cell clustering, having 
# largely replaced Louvain. It groups the cells on the neighborhood graph.
print(f"Running Leiden clustering with resolution {resolution}...")
sc.tl.leiden(adata, resolution=resolution)

In [ ]:
# ==============================================================================
# 5. Non-linear Dimensionality Reduction (UMAP)
# ==============================================================================
# UMAP takes the high-dimensional graph and squashes it flat into 2 dimensions 
# so we can actually look at it, while trying to preserve the global structure.
print("Calculating UMAP coordinates...")
sc.tl.umap(adata)

In [ ]:
# ==============================================================================
# 6. Visualization & Saving
# ==============================================================================
print(f"Plotting UMAP and saving to {figure_dir}...")
# Plot the UMAP and color the dots by the Leiden clusters we just found
sc.pl.umap(adata, color=['leiden'], title="PBMC 3k - Leiden Clusters", show=False, save="_pbmc3k_clusters.png")

print(f"Saving clustered data to {output_file}...")
adata.write(output_file)
print("Clustering and UMAP complete.")